In [ ]:
# ============================================================
# STATISTICAL WORD LENGTH VS FREQUENCY
# ============================================================
#
# This notebook illustrates the statistical approach to the
# coefficient word-length problem for digital filters.
#
# The central quantity is the statistical register length
#
#        L(omega) =
#        1 + B + log2[
#            x1 S(omega)
#            ------------------------------
#            sqrt(12) DeltaM_max
#        ]
#
# where
#
#       B
#           is the number of bits required for the integer part,
#
#       x1
#           is the statistical confidence factor,
#
#       S(omega)
#           is the combined sensitivity measure,
#
#       DeltaM_max
#           is the maximum admissible magnitude-response error.
#
#
# ============================================================
# STATISTICAL INTERPRETATION
# ============================================================
#
# The coefficient quantization error is modeled as a random
# variable uniformly distributed in the interval
#
#                  -Q/2 <= Delta c_k <= Q/2
#
# with
#
#                  E{Delta c_k} = 0
#
# and
#
#                  var{Delta c_k} = Q^2 / 12.
#
# If the coefficient errors are assumed uncorrelated, then
#
#                  sigma_DeltaM^2
#
#                    = Q^2/12 * S^2(omega),
#
# where
#
#                  S^2(omega)
#
#                    = sum_k
#                      [dM(e^jw)/dc_k]^2.
#
# For a sufficiently large number of coefficients, DeltaM may be
# approximated by a Gaussian random variable.
#
# A statistical error limit can then be written as
#
#                  DeltaM_1
#
#                    = x1 sigma_DeltaM.
#
# Requiring
#
#                  DeltaM_1 <= DeltaM_max
#
# leads to the quantization-step condition
#
#                  Q <=
#
#                    sqrt(12) DeltaM_max
#                    -------------------
#                    x1 S(omega).
#
# Since
#
#                  Q = 2^(-A),
#
# the required statistical word length becomes
#
#                  L(omega)
#
#                    = 1 + B
#
#                      + log2[
#                          x1 S(omega)
#                          --------------------------
#                          sqrt(12) DeltaM_max
#                        ].
#
#
# ============================================================
# MODELING ASSUMPTION USED IN THIS NOTEBOOK
# ============================================================
#
# In a complete filter realization, the combined sensitivity
#
#       S^2(omega) =
#           sum_k [dM(e^jw)/dc_k]^2
#
# would be calculated directly from the actual filter
# coefficients and structure.
#
# In this notebook, S(omega) is instead represented by a set of
# illustrative frequency-dependent sensitivity profiles.
#
# This is a deliberate pedagogical simplification.
#
# The purpose is to isolate and visualize how sensitivity,
# confidence factor and allowable magnitude-response error affect
# the required statistical word length.
#
# Once the assumed S(omega) profile is specified, all subsequent
# quantities
#
#       Q_max
#       A
#       L(omega)
#       L_required
#
# are calculated directly from the statistical word-length
# equations.
#
# Therefore, the numerical results are physically meaningful for
# the ASSUMED sensitivity profile, although they do not correspond
# to one specific filter realization.
#
#
# ============================================================
# WHAT THE NOTEBOOK SHOWS
# ============================================================
#
# The required register length is generally frequency dependent
# because the sensitivity S(omega) is frequency dependent.
#
# The graph shows:
#
#       continuous curve
#           -> theoretical statistical word length L(omega)
#
#       horizontal dashed line
#           -> final integer register length
#
#                  L_required
#
#                    = ceil(max L(omega)).
#
#
# ============================================================
# WHAT TO OBSERVE
# ============================================================
#
# 1. Increasing the confidence factor x1 increases the required
#    word length.
#
# 2. Decreasing DeltaM_max imposes a stricter error specification
#    and therefore increases the required word length.
#
# 3. Frequencies for which S(omega) is large require more bits.
#
# 4. Although L(omega) varies with frequency, an actual digital
#    implementation normally uses one register length capable of
#    satisfying the requirement over the entire frequency range.
#
# 5. Therefore,
#
#                  L_required
#
#                    = ceil(max L(omega))
#
#    is the practical design value.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, Layout
from IPython.display import display


# ------------------------------------------------------------
# Sensitivity profiles
# ------------------------------------------------------------

def sensitivity_profile(omega, profile, scale):

    if profile == 'Smooth':

        S = 1.0 + 0.8 * np.sin(omega)**2

    elif profile == 'Resonant':

        center = 0.55 * np.pi

        width = 0.12 * np.pi

        S = 0.8 + 5.0 * np.exp(-((omega - center) / width)**2)

    else:

        S = 0.8 + 4.0 * (omega / np.pi)**4

    return scale * S


# ------------------------------------------------------------
# Statistical word length
# ------------------------------------------------------------

def statistical_word_length(omega, B, x1, delta_m_max, profile, sensitivity_scale):

    S = sensitivity_profile(omega, profile, sensitivity_scale)

    L = 1.0 + B + np.log2(x1 * S / (np.sqrt(12.0) * delta_m_max))

    return L, S


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.swl-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.swl-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.swl-assumption {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #d5c58a;
    border-left: 6px solid #b8860b;
    background: #fffaf0;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.swl-assumption-title {
    font-size: 13px;
    font-weight: bold;
    color: #8a6500;
    margin-bottom: 4px;
}

.swl-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.swl-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.swl-info {
    font-size: 13px;
    line-height: 1.50;
}

.swl-label {
    display: inline-block;
    min-width: 235px;
    font-weight: bold;
}

.swl-value {
    font-size: 14px;
    font-weight: bold;
    color: #1b3a57;
}

.swl-highlight {
    color: #176b34;
    font-weight: bold;
}

.swl-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 5px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="swl-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Statistical Word Length vs Frequency
    </div>

</div>
""")


# ------------------------------------------------------------
# Description and visible modeling assumption
# ------------------------------------------------------------

description_html = HTML("""
<div class="swl-root">

    <div class="swl-description">

        The statistical register length depends on the frequency-dependent
        sensitivity <b>S(ω)</b>, the admissible magnitude-response error
        <b>ΔM<sub>max</sub></b>, and the selected confidence factor
        <b>x₁</b>.<br><br>

        The continuous curve represents the theoretical value
        <b>L(ω)</b>. The dashed horizontal line shows the practical integer
        word length obtained from
        <b>ceil(max L(ω))</b>.

    </div>

    <div class="swl-assumption">

        <div class="swl-assumption-title">
            Modeling assumption used in this demonstration
        </div>

        In a complete filter design, the sensitivity would be calculated from

        <div style="text-align:center; margin:7px 0;">
            <b>
            S²(ω) = Σ<sub>k</sub>
            [∂M(e<sup>jω</sup>)/∂c<sub>k</sub>]².
            </b>
        </div>

        In this notebook, <b>S(ω) is represented by illustrative
        frequency-dependent sensitivity profiles</b> rather than being derived
        from one particular filter realization. This is a deliberate
        pedagogical simplification.<br><br>

        Once the assumed sensitivity profile has been specified, the
        quantities <b>Q<sub>max</sub></b>, <b>A</b>, <b>L(ω)</b> and
        <b>L<sub>required</sub></b> are calculated directly from the
        statistical word-length equations. Therefore, the displayed results
        are physically meaningful for the assumed sensitivity profile.

    </div>

</div>
""")


# ------------------------------------------------------------
# Dynamic summary
# ------------------------------------------------------------

summary_html = HTML()

summary_html.layout = Layout(
    width='610px',
    min_width='610px',
    overflow='visible'
)


# ------------------------------------------------------------
# Main plotting function
# ------------------------------------------------------------

def plot_statistical_word_length(B=2, x1=2.0, delta_m_max=0.02, profile='Resonant', sensitivity_scale=1.0):


    # --------------------------------------------------------
    # Frequency grid
    # --------------------------------------------------------

    omega = np.linspace(0.0, np.pi, 1200)


    # --------------------------------------------------------
    # Calculate sensitivity and word length
    # --------------------------------------------------------

    L, S = statistical_word_length(omega, B, x1, delta_m_max, profile, sensitivity_scale)


    # --------------------------------------------------------
    # Practical required integer word length
    # --------------------------------------------------------

    L_max = np.max(L)

    L_required = int(np.ceil(L_max))

    max_index = int(np.argmax(L))

    omega_max = omega[max_index]

    S_max = S[max_index]


    # --------------------------------------------------------
    # Minimum word length
    # --------------------------------------------------------

    L_min = np.min(L)


    # --------------------------------------------------------
    # Maximum admissible quantization step at the most
    # restrictive frequency
    # --------------------------------------------------------

    Q_max = np.sqrt(12.0) * delta_m_max / (x1 * S_max)


    # --------------------------------------------------------
    # Equivalent number of fractional bits
    # --------------------------------------------------------

    A_required = np.log2(1.0 / Q_max)


    # --------------------------------------------------------
    # Approximate Gaussian confidence interpretation
    # --------------------------------------------------------

    if abs(x1 - 1.0) < 0.05:

        confidence_text = "approximately 68%"

    elif abs(x1 - 2.0) < 0.05:

        confidence_text = "approximately 95%"

    elif abs(x1 - 3.0) < 0.05:

        confidence_text = "approximately 99.7%"

    else:

        confidence_text = "user-selected statistical factor"


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    summary_html.value = f"""
    <div class="swl-box">

        <div class="swl-title">
            Statistical Word-Length Data
        </div>

        <div class="swl-info">

            <span class="swl-label">Integer-part bits</span>
            B = {B}
            <br>

            <span class="swl-label">Confidence factor</span>
            x₁ = <span class="swl-value">{x1:.2f}</span>
            &nbsp; ({confidence_text})
            <br>

            <span class="swl-label">Maximum allowed error</span>
            ΔM<sub>max</sub> = {delta_m_max:.4f}
            <br>

            <span class="swl-label">Sensitivity profile</span>
            {profile}
            <br>

            <span class="swl-label">Sensitivity scale</span>
            {sensitivity_scale:.2f}
            <br>

            <span class="swl-label">Maximum sensitivity</span>
            S<sub>max</sub> = {S_max:.4f}
            <br>

            <span class="swl-label">Critical frequency</span>
            ω = {omega_max / np.pi:.3f}π
            <br>

            <span class="swl-label">Minimum L(ω)</span>
            {L_min:.3f} bits
            <br>

            <span class="swl-label">Maximum L(ω)</span>
            {L_max:.3f} bits
            <br>

            <span class="swl-label">Required register length</span>
            <span class="swl-highlight">
                L = {L_required} bits
            </span>
            <br>

            <span class="swl-label">Maximum admissible Q</span>
            Q = {Q_max:.6e}
            <br>

            <span class="swl-label">Equivalent fractional bits</span>
            A ≈ {A_required:.3f}

        </div>

        <div class="swl-note">
            The practical register length is determined by the most
            restrictive frequency:
            L<sub>required</sub> = ceil(max L(ω)).
        </div>

    </div>
    """


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(12.5, 4.6)
    )


    ax.plot(
        omega,
        L,
        linewidth=2.0,
        label='Statistical word length L(ω)'
    )


    ax.axhline(
        L_required,
        linestyle='--',
        linewidth=1.6,
        label=f'Required integer length = {L_required} bits'
    )


    ax.plot(
        omega_max,
        L_max,
        'o',
        markersize=7,
        label='Most restrictive frequency'
    )


    # --------------------------------------------------------
    # Axis formatting
    # --------------------------------------------------------

    ax.set_xlim(
        0.0,
        np.pi
    )


    y_lower = max(
        0.0,
        np.floor(L_min) - 1.0
    )


    y_upper = np.ceil(
        max(
            L_required + 1.0,
            L_max + 1.0
        )
    )


    ax.set_ylim(
        y_lower,
        y_upper
    )


    ax.set_xticks(
        [
            0.0,
            np.pi / 4.0,
            np.pi / 2.0,
            3.0 * np.pi / 4.0,
            np.pi
        ]
    )


    ax.set_xticklabels(
        [
            '0',
            'π/4',
            'π/2',
            '3π/4',
            'π'
        ]
    )


    ax.set_xlabel(
        'Frequency ω'
    )


    ax.set_ylabel(
        'Required word length [bits]'
    )


    ax.set_title(
        'Frequency Dependence of the Statistical Register Length',
        fontsize=12
    )


    ax.grid(
        True,
        linestyle=':',
        alpha=0.5
    )


    # --------------------------------------------------------
    # Critical-frequency annotation
    # --------------------------------------------------------

    ax.annotate(
        f'max L(ω) = {L_max:.2f} bits',
        xy=(omega_max, L_max),
        xytext=(omega_max + 0.08 * np.pi, L_max - 1.0),
        arrowprops=dict(arrowstyle='->'),
        fontsize=9
    )


    # --------------------------------------------------------
    # Legend below horizontal axis
    # --------------------------------------------------------

    ax.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.17),
        ncol=3,
        fontsize=8,
        frameon=False
    )


    # --------------------------------------------------------
    # Final spacing
    # --------------------------------------------------------

    plt.subplots_adjust(
        left=0.08,
        right=0.98,
        top=0.90,
        bottom=0.25
    )


    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(
    width='310px'
)


slider_style = {
    'description_width': '145px'
}


integer_bits_slider = IntSlider(
    value=2,
    min=0,
    max=6,
    step=1,
    description='Integer bits B:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


confidence_slider = FloatSlider(
    value=2.0,
    min=1.0,
    max=3.0,
    step=0.1,
    description='Confidence factor x₁:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.1f'
)


error_slider = FloatSlider(
    value=0.02,
    min=0.002,
    max=0.10,
    step=0.002,
    description='Allowed error ΔM:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.3f'
)


sensitivity_slider = FloatSlider(
    value=1.0,
    min=0.25,
    max=4.0,
    step=0.05,
    description='Sensitivity scale:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


profile_selector = RadioButtons(
    options=[
        'Smooth',
        'Resonant',
        'Edge-sensitive'
    ],
    value='Resonant',
    description='S(ω) profile:',
    style={'description_width': '95px'},
    layout=Layout(
        width='310px'
    )
)


# ------------------------------------------------------------
# Interactive object
# ------------------------------------------------------------

widget_plot = interactive(
    plot_statistical_word_length,
    B=integer_bits_slider,
    x1=confidence_slider,
    delta_m_max=error_slider,
    profile=profile_selector,
    sensitivity_scale=sensitivity_slider
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls = VBox(
    [
        HTML("<div class='swl-title'>Controls</div>"),
        integer_bits_slider,
        confidence_slider,
        error_slider,
        sensitivity_slider,
        HTML("<div style='height:5px;'></div>"),
        profile_selector
    ],
    layout=Layout(
        width='335px',
        min_width='335px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Summary + controls
# ------------------------------------------------------------

top_row = HBox(
    [
        summary_html,
        controls
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Plot output
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ------------------------------------------------------------
# Final notebook layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)